# Week 3 Exercise - Synthetic Data Generator

Write models that can generate datasets. Use a variety of models and prompts for diverse outputs.

Create a Gradio UI for your product.

In [1]:
# imports

import os
import csv
import io
from openai import OpenAI
import google.generativeai as genai
from dotenv import load_dotenv
import gradio as gr

In [10]:
# constants

MODEL_GPT   = 'gpt-5-nano'
MODEL_GEMINI = 'gemini-2.0-flash'
MODEL_OPTIONS = [MODEL_GPT, MODEL_GEMINI]

COLUMNS = ["Produce Name", "Date", "Market Location", "Unit", "Price (USD)", "Season"]

In [3]:
# set up environment

load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')

if not openai_api_key:
    print("❌ OpenAI API key not found")
else:
    print("✅ OpenAI API key found!")
    openai_client = OpenAI(api_key=openai_api_key)

if not gemini_api_key:
    print("❌ Gemini API key not found")
else:
    print("✅ Gemini API key found!")
    genai.configure(api_key=gemini_api_key)

✅ OpenAI API key found!
✅ Gemini API key found!


In [4]:
# system prompt

system_prompt = """
You are a synthetic data generator specializing in agricultural market data.
Generate realistic market price records for produce.
Always return data as a CSV with exactly these columns in this order:
Produce Name, Date, Market Location, Unit, Price (USD), Season

Rules:
- Dates should be in YYYY-MM-DD format
- Prices should be realistic and vary by produce and season
- Locations should be real city names
- Units should be one of: kg, lb, bunch, dozen
- Season should be one of: Spring, Summer, Autumn, Winter
- Return ONLY the CSV rows, no headers, no explanation, no markdown
"""


In [5]:
# generate dataset function

def generate_dataset(model, num_rows):
    prompt = f"Generate {num_rows} rows of agricultural market price data for produce."

    if model == MODEL_GPT:
        response = openai_client.chat.completions.create(
            model=MODEL_GPT,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": prompt}
            ]
        )
        raw = response.choices[0].message.content.strip()

    elif model == MODEL_GEMINI:
        gemini = genai.GenerativeModel(
            model_name=MODEL_GEMINI,
            system_instruction=system_prompt
        )
        response = gemini.generate_content(prompt)
        raw = response.text.strip()

    # Parse into rows and build CSV
    rows = [line for line in raw.splitlines() if line.strip()]
    output = io.StringIO()
    writer = csv.writer(output)
    writer.writerow(COLUMNS)
    for row in rows:
        writer.writerow([col.strip() for col in row.split(",")])

    # Save to the current working directory
    filepath = os.path.join(os.getcwd(), "produce_market_data.csv")
    with open(filepath, "w") as f:
        f.write(output.getvalue())

    return output.getvalue(), filepath

In [12]:
# Gradio UI

with gr.Blocks() as ui:
    gr.Markdown("## 🌾 Agricultural Market Price Data Generator")
    gr.Markdown("""
    Welcome! This tool generates **synthetic agricultural market price data** for produce using AI.
    
    - Select a model to generate the data with
    - Choose how many rows you need (up to 100)
    - Hit **Generate Dataset** to create your data
    - Preview the results and download as a CSV file
    """)

    with gr.Row():
        model_picker = gr.Dropdown(
            choices=MODEL_OPTIONS,
            value=MODEL_GPT,
            label="Choose a model"
        )
        num_rows = gr.Slider(
            minimum=1,
            maximum=100,
            value=10,
            step=1,
            label="Number of rows"
        )

    generate_btn = gr.Button("Generate Dataset", variant="primary")

    csv_preview = gr.Textbox(
        label="CSV Preview",
        lines=15,
        interactive=False
    )
    download_btn = gr.File(label="Download CSV")

    generate_btn.click(
        fn=lambda: gr.update(interactive=False),
        outputs=generate_btn
    ).then(
        fn=generate_dataset,
        inputs=[model_picker, num_rows],
        outputs=[csv_preview, download_btn]
    ).then(
        fn=lambda: gr.update(interactive=True),
        outputs=generate_btn
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
